In [1]:
import os
import glob
import pandas as pd
import xlrd
import re
from pathlib import Path
import numpy as np
import xlsxwriter
import openpyxl
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from openpyxl.styles import Alignment, Border, Side, PatternFill

In [2]:
def format_excel(excel_path):
    
    wb = load_workbook(excel_path)
    #ws = wb.active

    for ws in wb.worksheets:
        wb.active = ws
        # filling colors
        season_colors = {
            'Kharif': 'FFFFFFDD',   # light yellow
            'Rabi': 'FFE7FFFF'      # light cyan
        }
        header_row = 1
        max_col = ws.max_column
        col_season = {}
        current_season = None
    
        for col_idx in range(1, max_col + 1):
            cell_value = ws.cell(row=header_row, column=col_idx).value
            if cell_value is not None and cell_value in season_colors:
                current_season = cell_value
            # If the cell is blank, it belongs to the previous season (if any)
            if current_season is not None:
                col_season[col_idx] = current_season
    
        for col_idx, season in col_season.items():
            fill = PatternFill(start_color=season_colors[season],
                               end_color=season_colors[season],
                               fill_type='solid')
            col_letter = get_column_letter(col_idx)
            for cell in ws[col_letter]:
                cell.fill = fill
    
        # making border
        thin_border = Border(
            left=Side(style='thin'),
            right=Side(style='thin'),
            top=Side(style='thin'),
            bottom=Side(style='thin')
        )
        
        for row in ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=1, max_col=ws.max_column):
            for cell in row:
                cell.border = thin_border
    
        # rotate row 2
        rotation_alignment = Alignment(textRotation=90, horizontal='center', vertical='bottom')
        for cell in ws[2]:
            cell.alignment = rotation_alignment
        
        # Set column widths (add a little padding)
        for col_index in range(1, ws.max_column+1):
            col_letter = get_column_letter(col_index)
            if col_index == 2:
                ws.column_dimensions[col_letter].width = 20
            else:
                ws.column_dimensions[col_letter].width = 3.6
    
    wb.save(excel_path)

In [3]:
def reorder_columns(df, season_crop_order):
    if df.empty:
        return df

    current_cols = df.columns
    existing = set(current_cols)
    type_order = ['UI', 'IR']  # UI first, then IR

    new_order = []
    for season in ['Kharif', 'Rabi']:   # Keep this order for seasons
        crop_list = season_crop_order.get(season, [])
        for crop in crop_list:
            for ctype in type_order:
                if (season, crop, ctype) in existing:
                    new_order.append((season, crop, ctype))
        season_cols = [col for col in current_cols if col[0] == season]
        ordered_crops = set(crop_list)
        remaining_crops = sorted([col for col in season_cols if col[1] not in ordered_crops],
                                 key=lambda x: x[1])  # alphabetical by crop
    # Build mapping for crop order per season
    crop_rank = {}
    for season, crops in season_crop_order.items():
        crop_rank[season] = {crop: idx for idx, crop in enumerate(crops)}

    type_rank = {t: i for i, t in enumerate(type_order)}

    def sort_key(col):
        season, crop, ctype = col
        # Season: Kharif first, Rabi second (ensure order)
        season_rank = 0 if season == 'Kharif' else 1
        # Crop rank: if in custom list, use its index; else use large number + alphabet
        crop_rank_for_season = crop_rank.get(season, {})
        if crop in crop_rank_for_season:
            crop_rank_val = crop_rank_for_season[crop]
        else:
            crop_rank_val = 1000 + ord(crop[0]) if crop else 999  # fallback
        # Type rank: UI=0, IR=1, others alphabetically after that
        if ctype in type_rank:
            type_rank_val = type_rank[ctype]
        else:
            type_rank_val = 10 + ord(ctype[0]) if ctype else 999
        return (season_rank, crop_rank_val, type_rank_val)

    sorted_cols = sorted(current_cols, key=sort_key)
    return df[sorted_cols]

In [4]:
def get_first_row(worksheet, row_text_list):
    for text in row_text_list:
        for row in worksheet.iter_rows():
            for cell in row:
                if cell.value and text.lower() in str(cell.value).lower():
                    return cell.row
    return -1

In [5]:
def get_first_col(worksheet, col_text_list):
    for text in col_text_list:
        for row in worksheet.iter_rows():
            for cell in row:
                if cell.value and text.lower() in str(cell.value).lower():
                    return cell.column
    return -1

In [6]:
def get_SL_to_WP(directory, st):
    records = []

    distt_seq = {
        "Kachchh": 1,
        "Surendranagar": 2,
        "Gandhinagar": 3,
        "Ahmedabad": 4,
        "Rajkot": 5,
        "Morabi": 6,
        "Junagadh": 7,
        "Gir Somnath": 8,
        "Jamnagar": 9,
        "Devbhumi Dwarka": 10,
        "Porbandar": 11,
        "Amreli": 12,
        "Bhavnagar": 13,
        "Botad": 14,
        "Banaskantha": 15,
        "Patan": 16,
        "Mahesana": 17,
        "Sabarkantha": 18,
        "Aravali": 19,
        "Anand": 20,
        "Kheda": 21,
        "Panchmahal": 22,
        "Mahisagar": 23,
        "Dahod": 24,
        "Vadodra": 25,
        "Chhota Udaipur": 26,
        "Narmada": 27,
        "Bharuch": 28,
        "Surat": 29,
        "Tapi": 30,
        "Dangs": 31,
        "Navsari": 32,
        "Valsad": 33
    }
    
    # Get all .xlsx and .xls files in the directory
    excel_files = glob.glob(os.path.join(directory, "*.xlsx"))
    df = pd.DataFrame()
    df_pivot = pd.DataFrame()
    df_long = pd.DataFrame()
    
    for file_path in excel_files:
        file_name = os.path.basename(file_path)
        distt = Path(file_name).stem.title()
        # Determine file type and use appropriate library
        if file_path.endswith('.xlsx'):
            try:
                wb = load_workbook(file_path, data_only=True)
                for sheet_name in wb.sheetnames:
                    if "cent" in sheet_name.lower() or "sta" in sheet_name.lower():
                        ws = wb[sheet_name]
                        
                        # setting which cells to scan
                        row_text_list = ['Taluka', 'Circle', 'VILLAGE', 'Exp', 'OS']
                        start_row = 0
                        title_row = 0
                        while start_row <= 0:
                            title_row = get_first_row(ws, row_text_list)
                            start_row = title_row + 1
                            
                        col_text_list = ['VILL']
                        start_col = 0
                        while start_col <= 0:
                            start_col = get_first_col(ws, col_text_list) + 1
                            taluka_col = start_col - 3
                            circle_col = start_col - 2
                            village_col = start_col - 1
                        
                        end_row = ws.max_row + 1
                        end_col = ws.max_column + 1

                        if start_row > 0 and start_col > 0:
                            for col_idx in range(start_col, end_col):
                                # get crop name and type
                                crop = ws.cell(row=start_row-2, column=col_idx).value
                                if isinstance(crop, str) and crop is not None:
                                    crop = crop.strip().replace(' ','')
                                else:
                                    crop = ws.cell(row=start_row-2, column=col_idx-1).value
                                    if isinstance(crop, str) and crop is not None:
                                        crop = crop.strip().replace(' ','')
                                    else:
                                        crop = None
    
                                # set season
                                if crop is not None:
                                    crop = re.sub(r'\s+', '', crop)
                                    crop = re.sub(r'UI', '', crop)
                                    
                                    plan = 0
                                    for row_idx in range(start_row, end_row):
                                        cell_val = ws.cell(row=row_idx, column=col_idx).value
                                        cell_header = ws.cell(row=title_row, column=col_idx).value
                                        
                                        if cell_val is not None:
                                            if taluka_col>0 and circle_col>0 and village_col>0:
                                                taluka = ws.cell(row=row_idx, column=taluka_col).value
                                                circle = ws.cell(row=row_idx, column=circle_col).value
                                                village = ws.cell(row=row_idx, column=village_col).value

                                                if taluka is not None and circle is not None and village is not None:
                                                    if 'exp' in str(cell_header).lower():
                                                        plan += cell_val
                                                    if 'os' in str(cell_header).lower():
                                                        selorder = cell_val
                                                        new_long_row = {
                                                            'SEASON': None,
                                                            'SAMPLE': st[0],
                                                            'DISTRICT': distt,
                                                            'OS': selorder,
                                                            'VILLAGE': village,
                                                            'NO_OF_EXPT': 2
                                                        }
                                                        df_long = pd.concat([df_long, pd.DataFrame([new_long_row])], ignore_index=True)
                                                        
                                                
                                    if plan != 0:
                                        new_row = {
                                            'CROP': crop,
                                            'CT': None,
                                            'SEASON': None,
                                            'DISTT': distt,
                                            'PLAN': plan,
                                            'SEQ': distt_seq[distt]
                                        }
                                        df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
                                        df = df.sort_values(['SEQ','SEASON','CROP','CT'])
                                        df_pivot = df.pivot_table(index=['SEQ','DISTT'],
                                                                  columns=['CROP'],
                                                                  values='PLAN',
                                                                  fill_value=np.nan,
                                                                  aggfunc='sum')
                                        df_pivot = df_pivot.sort_index()
                                        df_pivot = df_pivot.sort_index(axis=1, level=[0, 1, 2])

            except Exception as e:
                print(f"Error reading {file_name}, {st[0]}, {crop}, {plan} (openpyxl): {e}")
                raise e
    return df_pivot, df_long

In [7]:
if __name__ == "__main__":
    curr_dir = Path.cwd()
    directory = curr_dir / "SL 2.0"
    excel_path = curr_dir / "SL_to_WP.xlsx"

    # ----- Define season‑specific crop orders -----
    CROP_ORDER = {
        'Kharif': ['maize', 'groundnut', 'paddy(i)', 'paddy(ui)', 'sesamum',
                    'bajra', 'castor', 'tur', 'cotton(i)', 'cotton(ui)'],
        'Rabi': ['r&m', 'gram', 'wheat(i)'],
        'Summer': ['bajra']
    }

    with pd.ExcelWriter(excel_path, engine='xlsxwriter') as writer:
        for sample in range(1, 3):
            if sample == 1:
                st = 'cent'
            else:
                st = 'sta'

            df_pivot, df_long = get_SL_to_WP(directory, st)
            #df_pivot = reorder_columns(df_pivot, CROP_ORDER)   # Apply season‑specific order
            df_pivot.to_excel(writer, sheet_name=st)
        
        df_long.to_excel(writer, sheet_name="ML_data")

    # format excel workbook
    #format_excel(excel_path)